In [1]:
import pandas as pd

In [3]:
df_rera = pd.read_parquet("../data/clean_rer_a_2026-11-25.parquet")
df_rerb = pd.read_parquet("../data/clean_rer_b_2026-11-25.parquet")
df_rerd = pd.read_parquet("../data/clean_rer_d_2026-11-25.parquet")
df_incidents = pd.read_parquet("../data/clean_incidents_2026-11-25.parquet")

In [4]:
df_rera.columns

Index(['journey_id', 'line', 'mission', 'origin', 'destination', 'direction',
       'is_long', 'nb_stops', 'total_delay_sec', 'max_delay_added_sec',
       'worst_stop', 'disappeared', 'first_seen', 'last_seen', 'date',
       'stop_name', 'aimed_time', 'expected_time', 'delay_sec', 'platform',
       'status', 'stop_order', 'delay_added_sec', 'retard_min', 'est_supprime',
       'zone_tunnel', 'heure_pointe', 'tranche_horaire'],
      dtype='str')

In [5]:
# Axe 1 : comparaison globale des retards B/D vs A (hors suppressions, car IDFM laisse souvent delay_sec = 0)
# RER A : mean 1,28 min, P95 7,10 min ; RER B : 3,29 min, P95 12,75 min ; RER D : 3,08 min, P95 13,52 min

df_all = pd.concat([df_rera, df_rerb, df_rerd], ignore_index=True)
df_valide = df_all[~df_all["est_supprime"]]

macro = df_valide.groupby("line")["retard_min"].describe(percentiles=[0.5, 0.9, 0.95])
macro = macro[["count", "mean", "50%", "90%", "95%"]].rename(
    columns={"50%": "median", "90%": "p90", "95%": "p95"}
)
macro

,count,mean,median,p90,p95
line,,,,,
rer_a,147438.0,1.284295,0.366667,4.316667,7.100000
rer_b,259026.0,3.291276,1.483333,8.500000,12.750000
rer_d,57710.0,3.079007,0.933333,7.700000,13.516667


In [6]:
# Même comparaison sur Châtelet-Les Halles uniquement (gare commune aux 3 lignes, sans biais de longueur de tronçon)
# Confirme le classement B et D nettement au-dessus de A

df_chatelet = df_valide[df_valide["stop_name"] == "Châtelet-Les Halles"]
macro_chatelet = df_chatelet.groupby("line")["retard_min"].describe(percentiles=[0.5, 0.9, 0.95])
macro_chatelet = macro_chatelet[["count", "mean", "50%", "90%", "95%"]].rename(
    columns={"50%": "median", "90%": "p90", "95%": "p95"}
)
macro_chatelet

,count,mean,median,p90,p95
line,,,,,
rer_a,29812.0,1.147965,0.183333,4.183333,6.950000
rer_b,23164.0,3.206289,1.583333,8.350000,12.530833
rer_d,10816.0,3.426771,1.350000,8.125000,14.033333


In [7]:
# Retard ajouté à chaque arrêt (delay_added_sec, en secondes) hors tunnel vs dans le tunnel
# Hors tunnel les 3 lignes sont autour de +7 s (médiane 0 s)
# Dans le tunnel, le RER A rattrape (-9,3 s) tandis que B (+32,4 s, P90 +117 s) et D (+41,9 s, P90 +129 s) prennent du retard

df_valide.groupby(["line", "zone_tunnel"])["delay_added_sec"].agg(
    mean="mean",
    median="median",
    p90=lambda s: s.quantile(0.90),
)

mean  median    p90
line  zone_tunnel                          
rer_a False         7.266939     0.0   22.0
      True         -9.265665   -10.0    0.0
rer_b False         6.813791     0.0   40.0
      True         32.429026    11.0  117.0
rer_d False         7.872187     0.0   60.0
      True         41.874763    12.5  129.0

In [8]:
# Retard ajouté selon la direction (filtre stop_order > 1 pour exclure le 1er arrêt, et sens standards hors Aller/Retour)
# À Gare du Nord vers le Sud (entrée tunnel), le retard ajouté est nul/négatif (-1,4 s B, -3,8 s D)
# Il monte à la sortie du tunnel (Gare du Nord vers Nord : +43,3 s B, +48,6 s D ; Châtelet vers Sud : +45,3 s B, +82,3 s D)
# Le RER A reste négatif dans les deux sens

df_tunnel = df_valide[
    df_valide["zone_tunnel"]
    & (df_valide["stop_order"] > 1)
    & df_valide["direction"].isin(["Nord", "Sud", "Est", "Ouest"])
]

df_tunnel.groupby(["line", "direction", "stop_name"])["delay_added_sec"].agg(["mean", "median"])

mean  median
line  direction stop_name                             
rer_a Est       Châtelet-Les Halles -13.808970    -5.0
      Ouest     Châtelet-Les Halles  -5.467365   -10.0
rer_b Nord      Châtelet-Les Halles  14.235032    11.0
                Gare du Nord         43.304621    22.0
      Sud       Châtelet-Les Halles  45.340346    33.0
                Gare du Nord         -1.442104     0.0
rer_d Nord      Châtelet-Les Halles  34.568307     0.0
                Gare du Nord         48.588075    32.0
      Sud       Châtelet-Les Halles  82.319210    66.0
                Gare du Nord         -3.806086   -10.0

In [9]:
# Retard moyen (min) par tranche horaire
# Le RER A encaisse la pointe matin (2,78 min) mais redescend en pointe soir (1,46 min)
# Sur B et D, la pointe soir monte (5,45 min B, 4,30 min D) et déborde sur la soirée (3,37 et 3,71 min vs 1,46 min sur A)

df_valide.groupby(["line", "tranche_horaire"])["retard_min"].mean().unstack()

tranche_horaire,Heures creuses,Pointe matin,Pointe soir,Soirée,Week-end
line,,,,,
rer_a,1.180341,2.776894,1.461978,1.459590,0.369042
rer_b,2.806510,3.993930,5.449490,3.365350,2.342897
rer_d,2.734675,4.564714,4.300695,3.712309,1.533360


In [10]:
# Comparaison pointe vs heures creuses en semaine
# En ratio sur la moyenne (x1,61 à x1,76) les 3 lignes sont proches car le A part bas (1,18 min)
# L'écart se voit en valeur absolue (+1,98 min B, +1,68 min D vs +0,90 min A) et au P90 (11,5 et 11,0 min vs 6,2 min, ratio P90 de 1,72 sur D)

pointe = df_valide[df_valide["heure_pointe"]].groupby("line")["retard_min"].agg(
    pointe_mean="mean",
    pointe_p90=lambda s: s.quantile(0.90),
)
creuses = df_valide[df_valide["tranche_horaire"] == "Heures creuses"].groupby("line")["retard_min"].agg(
    creuses_mean="mean",
    creuses_p90=lambda s: s.quantile(0.90),
)

comp = pointe.join(creuses)
comp["ecart_mean"] = comp["pointe_mean"] - comp["creuses_mean"]
comp["ratio_mean"] = comp["pointe_mean"] / comp["creuses_mean"]
comp["ratio_p90"] = comp["pointe_p90"] / comp["creuses_p90"]
comp

,pointe_mean,pointe_p90,creuses_mean,creuses_p90,ecart_mean,ratio_mean,ratio_p90
line,,,,,,,
rer_a,2.077804,6.216667,1.180341,4.10,0.897463,1.760342,1.516260
rer_b,4.785965,11.500000,2.806510,7.35,1.979455,1.705308,1.564626
rer_d,4.411554,11.033333,2.734675,6.40,1.676879,1.613191,1.723958


In [11]:
# Retard ajouté dans le tunnel (sec) par tranche horaire
# Même le week-end et en soirée, B et D perdent +12 s à +29 s dans le tunnel contre -10 s à -13 s sur A
# En pointe soir, le A reste à -10,1 s pendant que B monte à +57,6 s et D à +68,2 s

df_tunnel.groupby(["line", "tranche_horaire"])["delay_added_sec"].mean().unstack()

tranche_horaire,Heures creuses,Pointe matin,Pointe soir,Soirée,Week-end
line,,,,,
rer_a,-10.647727,-0.704552,-10.113305,-10.445023,-12.679140
rer_b,20.871316,38.214481,57.577642,13.716403,12.211963
rer_d,37.183753,50.451653,68.173888,22.164286,29.066270
